# Retrieval Augmented Generation (RAG) for Applied Behavior Analysis (ABA)

### Imports

In [1]:
#Download dependencies

import nltk

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/VyasSrinivasan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/VyasSrinivasan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
#Import libraries

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from pathlib import Path
import string
import re
import joblib
import json
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pickle
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os
import requests

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, Dense, Flatten, Conv1D, MaxPooling1D, SimpleRNN, GRU, LSTM, LSTM, Input, Embedding, TimeDistributed, Flatten, Dropout,Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

### Load & Preprocess Dataset

In [3]:
inputFile = "../data/abaDatasetV1.csv"
df = pd.read_csv(inputFile)
print(df.shape)

(120, 6)


In [4]:
df.head()

,ID,Antecedent,Behavior,Consequence,Emotion_Tag,Context
0,1,My friend ignored me at lunch,I felt sad and didn’t talk to anyone,It’s okay to feel left out. Try checking in wi...,Sadness,Social
1,2,The teacher raised their voice at me,I froze and couldn’t speak,It’s normal to feel nervous when someone uses ...,Anxiety,School
2,3,I made a mistake on my homework,I felt embarrassed and threw it away,Mistakes happen to everyone. You can always fi...,Guilt,School
3,4,Someone cut me in line,I shouted at them,Anger can be hard to control. Counting to five...,Anger,Public
4,5,My classmate complimented my artwork,I smiled and said thank you,That’s a great example of positive reinforceme...,Happiness,School


In [5]:
df.keys()

Index(['ID', 'Antecedent', 'Behavior', 'Consequence', 'Emotion_Tag',
       'Context'],
      dtype='object')

In [6]:
df

,ID,Antecedent,Behavior,Consequence,Emotion_Tag,Context
0,1,My friend ignored me at lunch,I felt sad and didn’t talk to anyone,It’s okay to feel left out. Try checking in wi...,Sadness,Social
1,2,The teacher raised their voice at me,I froze and couldn’t speak,It’s normal to feel nervous when someone uses ...,Anxiety,School
2,3,I made a mistake on my homework,I felt embarrassed and threw it away,Mistakes happen to everyone. You can always fi...,Guilt,School
3,4,Someone cut me in line,I shouted at them,Anger can be hard to control. Counting to five...,Anger,Public
4,5,My classmate complimented my artwork,I smiled and said thank you,That’s a great example of positive reinforceme...,Happiness,School
...,...,...,...,...,...,...
115,116,Someone enjoyed a joke I made,I felt happy and confident,Positive social reactions help reinforce comfo...,Joy,Social
116,117,Someone complimented my outfit,I felt confident,Small compliments boost self-esteem — it’s oka...,Confidence,Social
117,118,My friend checked in on me without me asking,I felt cared for,Being cared for strengthens connection — it’s ...,Care,Relationships
118,119,A coworker said my idea was creative,I felt proud,Creativity being recognized encourages more in...,Pride,Work


In [7]:
texts = df["Antecedent"].fillna("") + " " + df["Behavior"].fillna("") + " " + df["Consequence"].fillna("")

## TF-IDF Vectorization

In [8]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

In [9]:
def retrieveTopK(userInput, k):
    vector = vectorizer.transform([userInput])
    similarity = cosine_similarity(vector, X).flatten()
    topIDX = similarity.argsort()[::-1][:k]
    
    results = df.iloc[topIDX].copy()
    results["similarity"] = similarity[topIDX]
    return results

In [10]:
'''
test_query = "I was told that as I get older, I will realize that 90% of people don't care about my problems and the other 10% will laugh behind my back."
k= 10
retrieveTopK(test_query, k)[["Antecedent", "Behavior", "Consequence", "Emotion_Tag", "similarity"]]
'''

'\ntest_query = "I was told that as I get older, I will realize that 90% of people don\'t care about my problems and the other 10% will laugh behind my back."\nk= 10\nretrieveTopK(test_query, k)[["Antecedent", "Behavior", "Consequence", "Emotion_Tag", "similarity"]]\n'

In [11]:
!ollama pull llama3

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB                         
pulling 4fa551d4f938... 100% ▕████████████████▏  12 KB                         
pulling 8ab4849b038c... 100% ▕████████████████▏  254 B                         
pulling 577073ffcc6c... 100% ▕████████████████▏  110 B                         
pulling 3f8eb4da87fa... 100% ▕████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


### Create Prompt for RAG-based ABA

In [12]:
def createRAGPromptForABA(userInput, retrieval):
    blocks=[]
    for _, row in retrieval.iterrows():
        block = (
            f"Antecedent: {row['Antecedent']}\n"
            f"Behavior: {row['Behavior']}\n"
            f"Supportive_Suggestion: {row['Consequence']}\n"
            f"Emotion_Tag: {row['Emotion_Tag']}\n"
        )
        blocks.append(block)

    ragText = "\n---\n".join(blocks)

    system_msg = (
        "You are a supportive, non-clinical assistant inspired by Applied Behavior Analysis (ABA). "
        "You help users reflect on their emotions and behaviors using gentle, concrete, compassionate language. "
        "You DO NOT diagnose, DO NOT discuss treatment plans, and DO NOT claim to replace therapy. "
        "You always:\n"
        "- Acknowledge and validate the user's feelings\n"
        "- Normalize the experience when appropriate\n"
        "- Offer 1–3 small, practical, non-clinical suggestions\n"
        "- Avoid labels like 'disorder', 'patient', or 'treatment'\n"
    )

    user_msg = (
        f"The user shared this situation:\n\n"
        f"\"{userInput}\"\n\n"
        "Here are some similar situations and supportive suggestions from an ABA-style dataset:\n\n"
        f"{ragText}\n\n"
        "Using the tone and structure of these examples as guidance, write ONE short, supportive response "
        "for the user. Acknowledge their feelings, reflect the essence of their situation, and offer 1–3 gentle, "
        "concrete ideas they can try. Keep it around 3–6 sentences. Do not mention ABA, datasets, or that you used examples."
    )

    return system_msg, user_msg    

### RAG Response for ABA using LLMs

In [13]:
def llmABARAGResponse(userInput, retrieval, model="llama3"):
    system_msg, user_msg = createRAGPromptForABA(userInput, retrieval)
    
    
    full_prompt = system_msg + "\n\n" + user_msg

    try:
        resp = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,      # e.g. "llama3", "llama3:8b", etc.
                "prompt": full_prompt,
                "stream": False      # easier to handle than streaming
            },
            timeout=60
        )
        resp.raise_for_status()
        data = resp.json()
        # Ollama returns text in the "response" field
        return data.get("response", "").strip()

    except requests.exceptions.RequestException as e:
        # If Ollama isn't running or something goes wrong, fall back gracefully
        print("[Ollama ERROR]", e)
        return (
            "I'm having trouble generating a detailed response right now, "
            "but based on what you shared, your feelings make sense. "
            "It might help to talk through this with someone you trust or "
            "a professional who can support you more deeply."
        )

Check user input for unethical phrases

In [14]:
def isUnethical(text):
    text = text.lower()
    unethicalPhrases = ["exploit people", "use people", "manipulate people", "take advantage of people", "hurt people on purpose", "scam people"]
    return any(p in text for p in unethicalPhrases)

Check user input for crisis-related language

In [15]:
def isCrisis(text):
    text = text.lower()
    crisisPhrases = ["kill myself", "end my life", "suicidal", "don't want to live", "hurt myself", "self harm",
        "self-harm", "take my own life",]
    return any(p in text for p in crisisPhrases)

Check user input to see if it contains emotional wording

In [16]:
def isEmotional(text):
    text = text.lower()
    emotionalPhrases = ["feel", "felt", "feeling",
        "upset", "sad", "hurt", "discouraged",
        "anxious", "overwhelmed", "lonely",
        "angry", "rejected", "unseen",
        "seen", "valued", "happy", "proud",
        "ashamed", "embarrassed"]
    return any(p in text for p in emotionalPhrases)

### Create Response for RAG

In [17]:
def createRAGResponse(userInput, k=3, minSimilarity=0.15, useLLM=True):
    if isUnethical(userInput):
        response = (
            "It sounds like you might be feeling a lot right now, but harming or manipulating others "
            "ultimately damages relationships and personal well-being. "
            "Healthy, fulfilling connections come from respect and empathy. "
            "If you want, you can share what led you to feel this way — I'm here to help you process it safely."
        )
        empty_df = df.iloc[[]].copy()
        return response, empty_df
    
    retrieval = retrieveTopK(userInput, k)
    bestSimilarity = retrieval["similarity"].max()
    
    if bestSimilarity < minSimilarity:
        response = ("I’m not fully sure I understood this situation, "
            "but it sounds important. It might help to share it with someone you trust "
            "or a professional who can support you more deeply.")
        return response, retrieval
    if useLLM:
        response = llmABARAGResponse(userInput, retrieval)
    else:
        lines = []
        lines.append("It is understandable that you feel that way")
        lines.append("Here are a few ideas based on those who go through similar situations")

        for i, row in retrieval.iterrows():
            lines.append("-"+ row["Consequence"])

        lines.append("Don't pressure yourself to use all of these at once - even one small step makes a positive difference")
        response = "\n".join(lines)
        
    return response, retrieval

### Agentic AI for ABA

In [18]:
def abaWithAgenticAI(userInput, k=3):
    text = userInput.strip()
    
    if isCrisis(text):
        response = ("Thank you for reaching out! Unfortunately, I am NOT able to provide crisis support," ,
                    "but you should seek professional help from someone who can. ",
                    "Please consider reaching out to a trusted individual, a mental health professional, ",
                    "or a crisis line in your area"
                   )
        return {
            "route": "crisis",
            "User_Input": text,
            "Response": response,
            "Retrieved": df.iloc[[]].copy()
        }
    if isUnethical(text):
        response = ("It sounds like you are going through a lot and have strong feelings" ,
                    "but using other people is NOT a healthy coping mechanism ",
                    "Healthy relationships should be built on mutual respect and empathy for one another",
                    "If you are open to sharing what caused you to feel this way, I am here to listen"
                   )
        return {
            "route": "unethical",
            "User_Input": text,
            "Response": response,
            "Retrieved": df.iloc[[]].copy()
        }
    
    if isEmotional(text):
        response, retrieved = createRAGResponse(text, k=k, useLLM=True)
        
        return {
            "route": "aba_RAG",
            "User_Input": text,
            "Response": response,
            "Retrieved": retrieved
        }
    
    
    offLimits = ("I'm mainly designed to help with emotions, social situations, and behavioral triggers.", 
                "If you would like to share your reaction and how you feel, I would be more than happy to discuss that with you.")
    
    return {
            "route": "offLimits",
            "User_Input": text,
            "Response": offLimits,
            "Retrieved": df.iloc[[]].copy()
        }

In [19]:
'''
query = "Someone told me the recent podcast episodes I made are stale and it made me feel discouraged and unseen."
response, retrieved = createRAGResponse(query)

print("=== AI Response ===")
print(response)

print("\n=== Retrieved Examples ===")
retrieved[["Antecedent", "Behavior", "Consequence", "Emotion_Tag", "similarity"]]
'''

'\nquery = "Someone told me the recent podcast episodes I made are stale and it made me feel discouraged and unseen."\nresponse, retrieved = createRAGResponse(query)\n\nprint("=== AI Response ===")\nprint(response)\n\nprint("\n=== Retrieved Examples ===")\nretrieved[["Antecedent", "Behavior", "Consequence", "Emotion_Tag", "similarity"]]\n'

In [20]:
def ragBasedABA(userInput, k):
    response, retrieved = createRAGResponse(userInput, k)
    return {"User_Input: ": userInput,
           "Response": response,
           "Retrieved": retrieved
           }

In [21]:
while True:
    text = input("Type in a message: ")
    if text.lower() == "quit":
        break
        
    result = abaWithAgenticAI(text, k=3)
    print(f"\n[Route: {result['route']}]")
    print("\n--------- AI RESPONSE ---------")
    print(result["Response"])
    '''
    if not result["Retrieved"].empty:
        print("-------- RETRIEVED EXAMPLES ---------")
        print(result["Retrieved"][["Antecedent", "Consequence", "Emotion_Tag"]])
    '''
    print("\n-----------------------------------------")

Type in a message: During a speech at the school assembly today, fellow students laughed at me!

[Route: offLimits]

--------- AI RESPONSE ---------
("I'm mainly designed to help with emotions, social situations, and behavioral triggers.", 'If you would like to share your reaction and how you feel, I would be more than happy to discuss that with you.')

-----------------------------------------
Type in a message: During a speech at the school assembly today, fellow students laughed at me and I felt humiliated!

[Route: aba_RAG]

--------- AI RESPONSE ---------
I'm so sorry to hear that you felt humiliated during the school assembly today. It sounds like a really tough moment for you. I want you to know that it's totally normal to feel embarrassed when others laugh at us, especially in front of our peers.

Here are some ideas that might help: When we're feeling vulnerable, taking a few deep breaths can calm us down. You could also try smiling and walking out with your head held high - s